In [ ]:
# Initialize Spark session

from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()


products = spark.createDataFrame([
    (1, "Apple Juice",        "Beverages"),
    (2, "Orange Juice",       "Beverages"),
    (3, "Chocolate Bar",      "Snacks"),
    (4, "Potato Chips",       "Snacks"),
    (5, "Fresh Strawberries", "Fruits"),
    (6, "Sparkling Water",    "Beverages"),
], ["product_id", "name", "category"])

sales = spark.createDataFrame([
    (1, 1, 10, 20),
    (2, 1,  5, 10),
    (3, 2,  8, 16),
    (4, 3,  2,  4),
    (5, 4, 15, 30),
    (6, 4,  5, 10),
    (7, 6, 12, 24),
], ["sale_id", "product_id", "quantity", "revenue"])

inventory = spark.createDataFrame([
    (1, 50, "Warehouse A"),
    (2, 40, "Warehouse A"),
    (2, 20, "Warehouse B"),
    (3, 30, "Warehouse A"),
    (4, 20, "Warehouse A"),
    (4, 15, "Warehouse B"),
    (5, 10, "Warehouse A"),
], ["product_id", "stock", "warehouse"])

# Your logic goes here to create df_result

ttl_qty = sales.groupBy(sales.product_id).agg(sum("quantity").alias("total_quantity"))
ttl_revenue = sales.groupBy(sales.product_id).agg(sum("revenue").alias("total_revenue"))
ttl_inventory = inventory.groupBy(inventory.product_id).agg(sum("stock").alias("total_stock"))

df_final = products.join(ttl_qty , "product_id", "left").join(ttl_revenue , "product_id" , "left")\
.join(ttl_inventory , "product_id", "left")\
.select("product_id", "name" , "category" , "total_quantity","total_revenue",   "total_stock")\
.fillna(0).orderBy("product_id")

display(df_final )